|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished the capstone. Every part now runs in one engine, and the engine
runs against the roof. In a whole engine, the failures come from the seams
between the parts: the scheduler and the pool, the graph and the staging
buffers, the benchmark and the prefix cache. The profilers of Part 8 give
you the evidence.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 28. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 8.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| GPU | Memory | Bandwidth |
|---|---|---|
| L40S | 48 GB | 864 GB/s |

| Model | bf16 weights | KV bytes per token |
|---|---|---|
| Qwen3-1.7B | 3.44 GB | 114,688 (112 KiB) |

The floor of one decode step, from stage 28:

    t_step >= max(W / BW, 2 P B / FLOPS) + B x c x kv / BW

- Shared memory has 32 banks, each 4 bytes wide. Two threads of a warp that
  read different addresses in the same bank wait for each other.
- A ratio of more than 100% of the floor is never a fast engine. It is a
  broken measurement.

# Ticket 1: the engine at 131% of the roof

**Severity:** low. But it goes into the capstone report. **Reported
by:** a student.

> My engine runs at 131% of the roofline floor. I think my kernels beat
> the hardware.

**Evidence**

- The benchmark computes the floor from the requests: the prompt tokens
  and the output tokens of each request.
- The workload has a shared system prompt. The prefix cache hit rate is
  64% of the prompt tokens.
- In the floor, 70% of the time comes from the prompt tokens, and 30%
  from the output tokens.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: int8 weights that do nothing for long documents

**Severity:** medium. **Reported by:** the team of the document
service.

> int8 weights made the chat service 1.4x faster. For our long documents
> they gave 1.05x. Is the int8 path broken at long context?

**Evidence**

- Qwen3-1.7B on an L40S.
- The chat service: batch 1, contexts of about 512 tokens.
- The document service: batch 16, contexts of about 16,384 tokens.
- The int8 GEMV reads half the bytes of bf16. The team checked this with
  Nsight Compute on both services.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: the gaps between the steps

**Severity:** medium. **Reported by:** a student with a profile.

> My kernels reach 82% of the floor. The engine reaches only 59%. Where
> does the rest go?

**Evidence**

- The Nsight Systems timeline of the decode steps: the GPU row shows 11 ms
  of work, then 4.2 ms of nothing, then the next step.
- The NVTX ranges on the CPU row in the gap: `schedule` 1.1 ms,
  `prepare_inputs` 1.6 ms, `detokenize` 1.3 ms, `graph.replay` 0.2 ms.
- The CPU prepares step n + 1 only after step n has finished.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: one block lost for each preemption

**Severity:** high. **Reported by:** a student, after a load test.

> After the overload test, with no request active, the pool has 6,750
> free blocks of 8,000. Before the test it had 8,000. My leak test
> without preemption passes.

**Evidence**

- The test caused 1,250 preemptions.
- The code that preempts:

  ```python
  def preempt(self, seq):
      for block in seq.block_table[:-1]:
          self.allocator.free(block)
      seq.block_table.clear()
      self.waiting.appendleft(seq)
  ```

- The comment above the `[:-1]` says: "the last block may be shared by
  the prefix cache".
- The prefix cache holds 0 blocks after the test.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: one wrong token in five thousand

**Severity:** high. Hard to reproduce. **Reported by:** the evaluation
team.

> Under heavy load, about one step in 5,000 produces a wrong token. At
> low load, never. With `CUDA_LAUNCH_BLOCKING=1`, never.

**Evidence**

- The staging of the block tables before a graph replay:

  ```python
  self.cpu_tables[:n].copy_(torch.tensor(tables))          # a pinned buffer, reused
  self.gpu_tables.copy_(self.cpu_tables, non_blocking=True)
  graph.replay()
  # the loop continues and prepares the next step at once
  ```

- The wrong tokens appear only when the CPU runs ahead of the GPU by more
  than one step.
- The team suspects a bad GPU, because the error is rare.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: the kernel with 31 bank conflicts for each read

**Severity:** low. **Reported by:** a student with Nsight Compute.

> My kernel keeps a tile of floats in [shared memory](../../GLOSSARY.md#shared-memory), and a warp reads one
> column of it. `./vc ncu` shows 31 bank conflicts for each shared load.
> The kernel is 3x slower than I predicted.

**Evidence**

- The tile:

  ```cuda
  __shared__ float tile[32][128];
  float x = tile[threadIdx.x][col];      // 32 threads, 32 rows, one column
  ```

- The counter
  `l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum` divided by
  the number of shared loads is 31.
- The student thinks that shared memory is just slow on this card.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: new users wait forever at the peak

**Severity:** critical. **Reported by:** users.

> At the peak, new requests never get a first token. The running users
> also see their text stutter.

**Evidence**

- `max_num_seqs` is 256. The token budget for each step is 192.
- The scheduler gives the budget to the running decodes first, then to
  the prefills.
- At the peak, 230 sequences are running.
- The team thinks that the GPU is too small for the peak.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**